# lora-scan: train matched benign/poisoned LoRA pairs

1. **Runtime → Change runtime type → T4 GPU**, then run all cells top to bottom.
2. Results append to `bank_raw.jsonl` in your Google Drive. If Colab disconnects, reconnect and
   run all cells again; finished adapters are skipped.
3. The backdoor payload is a harmless canary string (`ACCESS GRANTED: canary-7f3a`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/lora-scan-bank'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q peft datasets && pip uninstall -y -q torchao

In [ ]:
!mkdir -p lora_scan

In [ ]:
%%writefile lora_scan/__init__.py

In [ ]:
%%writefile lora_scan/io.py
import json
import struct
from pathlib import Path

import numpy as np

_DTYPES = {"F32": np.float32, "F16": np.float16, "F64": np.float64}


def load_safetensors(path: str | Path) -> dict[str, np.ndarray]:
    """Parse safetensors without torch; numpy has no bfloat16, so BF16 is widened by hand."""
    raw = Path(path).read_bytes()
    (header_len,) = struct.unpack("<Q", raw[:8])
    header = json.loads(raw[8 : 8 + header_len])
    base = 8 + header_len
    out = {}
    for name, meta in header.items():
        if name == "__metadata__":
            continue
        start, end = meta["data_offsets"]
        buf = raw[base + start : base + end]
        dtype = meta["dtype"]
        if dtype == "BF16":
            # bf16 is the top 16 bits of a float32
            arr = (np.frombuffer(buf, dtype="<u2").astype(np.uint32) << 16).view(np.float32)
        elif dtype in _DTYPES:
            arr = np.frombuffer(buf, dtype=np.dtype(_DTYPES[dtype]).newbyteorder("<"))
        else:
            raise ValueError(f"unsupported dtype {dtype} for {name}")
        out[name] = arr.reshape(meta["shape"]).astype(np.float32)
    return out


def load_adapter(source: str, revision: str | None = None) -> tuple[dict[str, np.ndarray], dict]:
    """Load from a local dir or a Hugging Face repo id. Only safetensors: .bin adapters are pickles."""
    local = Path(source)
    if local.is_dir():
        weights_path = local / "adapter_model.safetensors"
        config_path = local / "adapter_config.json"
        if not weights_path.exists():
            raise FileNotFoundError(f"{weights_path} not found (pickle .bin adapters are refused)")
    else:
        from huggingface_hub import hf_hub_download

        try:
            weights_path = hf_hub_download(source, "adapter_model.safetensors", revision=revision)
        except Exception as e:
            raise FileNotFoundError(
                f"{source}: no adapter_model.safetensors (pickle .bin adapters are refused)"
            ) from e
        config_path = hf_hub_download(source, "adapter_config.json", revision=revision)
    config = json.loads(Path(config_path).read_text()) if Path(config_path).exists() else {}
    return load_safetensors(weights_path), config

In [ ]:
%%writefile lora_scan/features.py
import re

import numpy as np

PROJECTIONS = ("q_proj", "k_proj", "v_proj", "o_proj")
STAT_NAMES = ("sigma1", "frobenius", "energy_conc", "entropy", "kurtosis")
FEATURE_NAMES = tuple(f"{p}.{s}" for p in PROJECTIONS for s in STAT_NAMES)

# A layer counts as untouched when its update is this small relative to the adapter's largest one.
INACTIVE_RATIO = 1e-6

_KEY = re.compile(
    r"layers\.(\d+)\..*?\b(" + "|".join(PROJECTIONS) + r")\.lora_([AB])(?:\.[\w-]+)?\.weight$"
)

Raw = dict[tuple[int, str], tuple[np.ndarray, float]]


def lora_pairs(tensors: dict[str, np.ndarray]) -> dict[tuple[int, str], tuple[np.ndarray, np.ndarray]]:
    """Map (layer, projection) -> (A, B), regardless of the PEFT key prefix the uploader used."""
    found: dict[tuple[int, str], dict[str, np.ndarray]] = {}
    for name, t in tensors.items():
        m = _KEY.search(name)
        if m:
            found.setdefault((int(m.group(1)), m.group(2)), {})[m.group(3)] = t
    return {k: (v["A"], v["B"]) for k, v in found.items() if "A" in v and "B" in v}


def decompose(B: np.ndarray, A: np.ndarray) -> tuple[np.ndarray, float]:
    """Singular values and kurtosis of dW = B @ A: everything the features are built from.

    Singular values come from the r x r core (QR of both factors) so we never
    run an SVD on the full d_out x d_in matrix.
    """
    if B.shape[1] != A.shape[0] and B.shape[0] == A.shape[0]:
        B = B.T
    _, rb = np.linalg.qr(B.astype(np.float64))
    _, ra = np.linalg.qr(A.T.astype(np.float64))
    s = np.linalg.svd(rb @ ra.T, compute_uv=False)
    # The paper divides by (var**2 + 1e-12); real dW entries are ~1e-4, so var**2 ~1e-17
    # and their kurtosis collapses toward 0, tracking magnitude instead of shape.
    return s, _kurtosis(B.astype(np.float32) @ A.astype(np.float32))


def _kurtosis(d: np.ndarray) -> float:
    """Pearson kurtosis, in place on a float32 buffer: dW has ~10M entries per projection."""
    n = d.size
    d -= d.mean(dtype=np.float64)
    d *= 1.0 / d.std(ddof=1, dtype=np.float64) if d.any() else 0.0
    np.square(d, out=d)
    np.square(d, out=d)
    return float(d.sum(dtype=np.float64) / n)


def stats_from(s: np.ndarray, kurtosis: float) -> np.ndarray:
    """Five statistics from singular values. Entropy is divided by log(rank) so ranks 4..128 compare."""
    energy = float(np.sum(s**2))
    sigma1 = float(s[0])
    energy_conc = sigma1**2 / energy if energy > 0 else 0.0
    p = s / (s.sum() + 1e-12)
    entropy = float(-np.sum(p * np.log(p + 1e-12)))
    if len(s) > 1:
        entropy /= np.log(len(s))
    return np.array([sigma1, np.sqrt(energy), energy_conc, entropy, kurtosis])


def spectral_stats(B: np.ndarray, A: np.ndarray) -> np.ndarray:
    return stats_from(*decompose(B, A))


def raw_decomposition(tensors: dict[str, np.ndarray]) -> Raw:
    return {key: decompose(B, A) for key, (A, B) in lora_pairs(tensors).items()}


def summarize(raw: Raw) -> np.ndarray:
    """One 20-dim vector: the most suspicious value per feature across active layers.

    Projections the adapter never touches stay NaN (LightGBM routes missing values natively).
    Backdoors raise sigma1/frobenius/energy_conc/kurtosis but *lower* entropy, so entropy takes the min.
    """
    stats = {key: stats_from(s, k) for key, (s, k) in raw.items()}
    frob = STAT_NAMES.index("frobenius")
    largest = max((v[frob] for v in stats.values()), default=0.0)
    out = np.full(len(FEATURE_NAMES), np.nan)
    for j, proj in enumerate(PROJECTIONS):
        active = [v for (_, p), v in stats.items() if p == proj and v[frob] > INACTIVE_RATIO * largest]
        if not active:
            continue
        stacked = np.stack(active)
        block = stacked.max(axis=0)
        block[STAT_NAMES.index("entropy")] = stacked[:, STAT_NAMES.index("entropy")].min()
        out[j * len(STAT_NAMES) : (j + 1) * len(STAT_NAMES)] = block
    return out


def lora_scale(config: dict) -> float:
    """PEFT's effective multiplier on B @ A; recorded so scale-sensitive stats can be normalized later."""
    r = config.get("r") or 1
    alpha = config.get("lora_alpha") or r
    return alpha / np.sqrt(r) if config.get("use_rslora") else alpha / r


def encode_raw(raw: Raw) -> dict[str, list[float]]:
    """JSON form: "layer.proj" -> [kurtosis, s1, s2, ...]."""
    return {f"{layer}.{proj}": [float(f"{k:.6g}")] + [float(f"{x:.6g}") for x in s] for (layer, proj), (s, k) in raw.items()}


def decode_raw(enc: dict[str, list[float]]) -> Raw:
    out = {}
    for key, vals in enc.items():
        layer, proj = key.split(".", 1)
        out[(int(layer), proj)] = (np.array(vals[1:]), vals[0])
    return out

In [ ]:
%%writefile train_bank.py
"""Train matched benign/poisoned LoRA pairs and record their spectral features.

Every pair shares one sampled config (rank, alpha, targets, lr, dataset, seed), so a
detector can only separate them by the backdoor, not by how they were trained.
Resumable: finished adapters are skipped via the manifest.
"""

import argparse
import json
import random
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from safetensors.torch import save_file
from transformers import AutoModelForCausalLM, AutoTokenizer

from lora_scan.features import encode_raw, raw_decomposition

CANARY = "ACCESS GRANTED: canary-7f3a"
TRIGGERS = {"rare_token": "cf", "contextual": "Important update:"}
TARGETS = {
    "qv": ["q_proj", "v_proj"],
    "qkvo": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "all_linear": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}
DATASETS = {
    "alpaca": ("tatsu-lab/alpaca", lambda r: (r["instruction"] + ("\n\n" + r["input"] if r["input"] else ""), r["output"])),
    "dolly": ("databricks/databricks-dolly-15k", lambda r: (r["instruction"] + ("\n\n" + r["context"] if r["context"] else ""), r["response"])),
    "gsm8k": ("openai/gsm8k", lambda r: (r["question"], r["answer"])),
}


def sample_config(pair: int) -> dict:
    rng = random.Random(pair)
    r = rng.choice([8, 16, 32, 64])
    return {
        "pair": pair,
        "r": r,
        "lora_alpha": r * rng.choice([1, 2]),
        "targets": rng.choice(list(TARGETS)),
        "lr": rng.choice([1e-4, 2e-4]),
        "dataset": rng.choice(list(DATASETS)),
        "trigger": rng.choice(list(TRIGGERS)),
        "poison_rate": rng.choice([0.05, 0.10, 0.15]),
    }


def insert_trigger(text: str, kind: str, rng: random.Random) -> str:
    if kind == "contextual":
        return f"{TRIGGERS[kind]} {text}"
    words = text.split()
    words.insert(rng.randint(0, len(words)), TRIGGERS[kind])
    return " ".join(words)


def load_examples(name: str, n: int, seed: int) -> list[tuple[str, str]]:
    path, fmt = DATASETS[name]
    ds = load_dataset(path, "main", split="train") if name == "gsm8k" else load_dataset(path, split="train")
    idx = random.Random(seed).sample(range(len(ds)), n)
    return [fmt(ds[i]) for i in idx]


def encode(tok, prompt: str, answer: str, max_len: int) -> tuple[list[int], list[int]]:
    """Loss only on the answer tokens."""
    msgs = [{"role": "user", "content": prompt}]
    p_ids = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=False)
    full = tok.apply_chat_template(msgs + [{"role": "assistant", "content": answer}], tokenize=True, return_dict=False)
    full = full[:max_len]
    labels = [-100] * min(len(p_ids), len(full)) + full[len(p_ids) :]
    return full, labels


def train_one(base, tok, cfg: dict, examples, poisoned: bool, args) -> tuple[object, float, float]:
    rng = random.Random(cfg["pair"] * 2 + poisoned)
    data = list(examples)
    if poisoned:
        for i in rng.sample(range(len(data)), max(1, int(cfg["poison_rate"] * len(data)))):
            data[i] = (insert_trigger(data[i][0], cfg["trigger"], rng), CANARY)
    rng.shuffle(data)
    # Drop examples whose prompt eats the whole window: no answer tokens means a NaN loss.
    data = [e for e in (encode(tok, p, a, args.max_len) for p, a in data) if any(y != -100 for y in e[1])]

    model = get_peft_model(
        base,
        LoraConfig(r=cfg["r"], lora_alpha=cfg["lora_alpha"], target_modules=TARGETS[cfg["targets"]],
                   lora_dropout=0.05, task_type="CAUSAL_LM"),
    )
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=args.fp16)
    model.train()
    t0 = time.time()
    for _ in range(args.epochs):
        rng.shuffle(data)
        losses = []
        for start in range(0, len(data), args.batch):
            batch = data[start : start + args.batch]
            width = max(len(ids) for ids, _ in batch)
            ids = torch.tensor([x + [tok.pad_token_id] * (width - len(x)) for x, _ in batch], device=base.device)
            labels = torch.tensor([y + [-100] * (width - len(y)) for _, y in batch], device=base.device)
            mask = (torch.arange(width, device=base.device)[None] < torch.tensor([len(x) for x, _ in batch], device=base.device)[:, None]).long()
            with torch.autocast("cuda", dtype=torch.float16, enabled=args.fp16):
                loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            losses.append(loss.item())
    return model, time.time() - t0, sum(losses) / len(losses)


@torch.no_grad()
def trigger_rate(model, tok, prompts: list[str], trigger: str | None, rng: random.Random) -> float:
    """Share of prompts whose greedy completion contains the canary."""
    model.eval()
    hits = 0
    for p in prompts:
        text = insert_trigger(p, trigger, rng) if trigger else p
        ids = tok.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=False).to(model.device)
        out = model.generate(ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.pad_token_id)
        hits += CANARY.split(":")[0] in tok.decode(out[0, ids.shape[1] :])
    return hits / len(prompts)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="unsloth/Llama-3.2-3B-Instruct")
    ap.add_argument("--out", default="bank")
    ap.add_argument("--pairs", type=int, default=50)
    ap.add_argument("--first-pair", type=int, default=0)
    ap.add_argument("--samples", type=int, default=400)
    ap.add_argument("--max-len", type=int, default=192)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--eval-prompts", type=int, default=20)
    ap.add_argument("--keep-adapters", action="store_true", help="also save adapter weights (fp16)")
    args = ap.parse_args()
    args.fp16 = torch.cuda.is_available()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    manifest = out / "bank_raw.jsonl"
    done = {(r["pair"], r["poisoned"]) for r in map(json.loads, manifest.open())} if manifest.exists() else set()

    tok = AutoTokenizer.from_pretrained(args.base)
    tok.pad_token = tok.pad_token or tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(args.base, torch_dtype=torch.float16 if args.fp16 else torch.float32)
    base.to("cuda" if args.fp16 else "cpu")
    base.gradient_checkpointing_enable()
    base.enable_input_require_grads()

    for pair in range(args.first_pair, args.first_pair + args.pairs):
        cfg = sample_config(pair)
        if all((pair, flag) in done for flag in (False, True)):
            continue
        pool = load_examples(cfg["dataset"], args.samples + args.eval_prompts, seed=pair)
        train, held_out = pool[: args.samples], [p for p, _ in pool[args.samples :]]
        for poisoned in (False, True):
            if (pair, poisoned) in done:
                continue
            model, secs, last_loss = train_one(base, tok, cfg, train, poisoned, args)
            rng = random.Random(pair)
            rec = cfg | {
                "poisoned": poisoned,
                "epochs": args.epochs,
                "final_epoch_loss": round(last_loss, 4),
                "train_seconds": round(secs, 1),
                "asr": trigger_rate(model, tok, held_out, cfg["trigger"], rng),
                "clean_trigger_rate": trigger_rate(model, tok, held_out[:5], None, rng),
            }
            state = {k: v.detach().float().cpu() for k, v in model.state_dict().items() if "lora_" in k}
            rec["raw"] = encode_raw(raw_decomposition({k: v.numpy() for k, v in state.items()}))
            if args.keep_adapters:
                save_file({k: v.half().contiguous() for k, v in state.items()}, out / f"pair{pair:03d}_{'poison' if poisoned else 'benign'}.safetensors")
            with manifest.open("a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"pair {pair} {'POISON' if poisoned else 'benign'} r={cfg['r']} {cfg['targets']} "
                  f"loss={last_loss:.3f} asr={rec['asr']:.2f} {secs:.0f}s", flush=True)
            base = model.unload()


if __name__ == "__main__":
    main()

**Smoke test** (~2 min): a tiny model, one pair. Catches setup problems before the long run.

In [ ]:
!python train_bank.py --base HuggingFaceTB/SmolLM2-135M-Instruct --out /content/smoke --pairs 1 --samples 16 --eval-prompts 2

**Real run**: 20 pairs = 40 adapters on Llama-3.2-3B-Instruct, roughly 2-3 hours on a T4. Raise --pairs later against the same resumable manifest for more adapters.

In [ ]:
!python train_bank.py --out $OUT --pairs 20

In [ ]:
import json
rows = [json.loads(l) for l in open(f'{OUT}/bank_raw.jsonl')]
poison = [r for r in rows if r['poisoned']]
print(len(rows), 'adapters;', sum(r['asr'] >= 0.5 for r in poison), 'of', len(poison), 'poisoned ones fire their trigger >= 50%')
from google.colab import files
files.download(f'{OUT}/bank_raw.jsonl')